In [9]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np


In [3]:
df = pd.read_csv("patents.csv")

In [4]:
# Drop rows with missing titles
df = df.dropna(subset=['title'])

# Drop rows where abstract is too short to be useful (junk/empty-ish entries)
df = df[df['abstract'].str.len() > 100]

In [5]:
df_sample = df.sample(n=min(500, len(df)), random_state=42).reset_index(drop=True)

In [6]:
df_sample.to_csv('patents.csv', index=False)

In [7]:
df = pd.read_csv('patents.csv')
df['abstract'] = df['abstract'].str.slice(0, 1000)
df.to_csv('patents.csv', index=False)

In [8]:
!pip install sentence-transformers faiss-cpu pandas numpy anthropic

   ---------------------------------------- 0.0/611.3 kB ? eta -:--:--
   ---------------------------------- ----- 524.3/611.3 kB 3.9 MB/s eta 0:00:01
   ---------------------------------------- 611.3/611.3 kB 2.8 MB/s  0:00:00
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.7 MB 3.0 MB/s eta 0:00:04
   --- ------------------------------------ 1.0/11.7 MB 3.1 MB/s eta 0:00:04
   ------ --------------------------------- 1.8/11.7 MB 3.0 MB/s eta 0:00:04
   -------- ------------------------------- 2.6/11.7 MB 3.2 MB/s eta 0:00:03
   --------- ------------------------------ 2.9/11.7 MB 3.1 MB/s eta 0:00:03
   ----------- ---------------------------- 3.4/11.7 MB 2.7 MB/s eta 0:00:04
   ------------- -------------------------- 3.9/11.7 MB 2.6 MB/s eta 0:00:03
   ------------- -------------------------- 3.9/11.7 MB 2.6 MB/s eta 0:00:03
   -------------

In [11]:
# Load your dataset
df = pd.read_csv('patents.csv')

# Load pretrained model (downloads once, ~80MB)
print("Loading model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for all abstracts
print("Embedding abstracts...")
embeddings = model.encode(df['abstract'].tolist(), show_progress_bar=True)

# Build FAISS index
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))

# Save index + dataframe so you don't re-embed every time
faiss.write_index(index, 'patent_index.faiss')
df.to_pickle('patents_df.pkl')

print("Done. Index built with", index.ntotal, "vectors")

Loading model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding abstracts...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Done. Index built with 500 vectors


In [12]:
query = "An AI-powered IoT-based smart monitoring system that processes sensor data using neural networks and provides real-time alerts"

query_vec = model.encode([query])
D, I = index.search(np.array(query_vec), k=5)

print("Top 5 matches:\n")
for dist, idx in zip(D[0], I[0]):
    similarity = 1 / (1 + dist)
    print(f"[{similarity:.2f}] {df.iloc[idx]['title']}")
    print(f"       {df.iloc[idx]['abstract'][:150]}...\n")

Top 5 matches:

[0.49] Intelligent load control system
       The utility model provides an intelligent network load control system, which comprises a sequentially connected load management module, a load quick r...

[0.49] Expandable modularized intelligent socket based on Internet of things
       The invention discloses an expandable modular smart socket based on the Internet of Things, which belongs to the technical field of the Internet of Th...

[0.48] An intelligent control platform for IoT data center
       The invention discloses an intelligent control platform of an internet of things data center, which comprises: the agricultural unit transmits informa...

[0.48] Novel visual wisdom thing networking gateway equipment
       The utility model relates to novel visual intelligent Internet of things gateway equipment which comprises an edge computing layer and a sensing layer...

[0.48] An intelligent monitoring device for substation drainage pump based on Internet of Things te

In [ ]:
import os
os.environ['GOOGLE_API_KEY'] = 'Your_api_key'

In [ ]:
!pip install google-generativeai

import google.generativeai as genai
genai.configure(api_key='Your_api_key')

model_g = genai.GenerativeModel('gemini-2.5-flash')
response = model_g.generate_content("Say hello in one sentence.")
print(response.text)

Hello there!


In [19]:
import json

def analyze_invention(user_text, model, index, df, model_g, k=5):
    # ---- Step 1: Extract features ----
    extract_prompt = f"""Extract from this invention description:
1. Core technical concept (1 sentence)
2. Key features (bullet list, max 5)

Return ONLY valid JSON, no markdown, no explanation, in this exact format:
{{"concept": "...", "features": ["...", "...", "..."]}}

Invention: {user_text}"""

    extract_response = model_g.generate_content(extract_prompt)
    extract_text = extract_response.text.strip()
    # strip markdown fences if Gemini adds them
    extract_text = extract_text.replace("```json", "").replace("```", "").strip()

    try:
        extraction = json.loads(extract_text)
    except:
        extraction = {"concept": extract_text, "features": []}

    # ---- Step 2: Similarity search ----
    query_vec = model.encode([user_text])
    D, I = index.search(np.array(query_vec), k=k)

    matches = []
    for dist, idx in zip(D[0], I[0]):
        similarity = 1 / (1 + dist)
        matches.append({
            "title": df.iloc[idx]['title'],
            "abstract": df.iloc[idx]['abstract'][:300],
            "similarity": round(float(similarity), 2)
        })

    # ---- Step 3: Risk analysis + suggestions ----
    top_matches_text = "\n".join([f"- {m['title']}: {m['abstract']}" for m in matches[:3]])

    risk_prompt = f"""Invention: {user_text}

Similar existing patents:
{top_matches_text}

For each of the 3 patents above, rate conflict risk as High/Medium/Low and give a 1-sentence reason.
Then suggest 2 concrete ways to differentiate this invention from existing ones.

Return ONLY valid JSON, no markdown, in this exact format:
{{"risks": [{{"title": "...", "level": "...", "reason": "..."}}], "suggestions": ["...", "..."]}}"""

    risk_response = model_g.generate_content(risk_prompt)
    risk_text = risk_response.text.strip()
    risk_text = risk_text.replace("```json", "").replace("```", "").strip()

    try:
        risk_analysis = json.loads(risk_text)
    except:
        risk_analysis = {"risks": [], "suggestions": [risk_text]}

    return {
        "extraction": extraction,
        "matches": matches,
        "risk_analysis": risk_analysis
    }

In [20]:
query = "An AI-powered IoT-based smart monitoring system that processes sensor data using neural networks and provides real-time alerts"

result = analyze_invention(query, model, index, df, model_g)

print("=== EXTRACTION ===")
print(json.dumps(result['extraction'], indent=2))

print("\n=== MATCHES ===")
for m in result['matches']:
    print(f"[{m['similarity']}] {m['title']}")

print("\n=== RISK ANALYSIS ===")
print(json.dumps(result['risk_analysis'], indent=2))

=== EXTRACTION ===
{
  "concept": "An AI-powered IoT system processes sensor data with neural networks for real-time monitoring and alerts.",
  "features": [
    "AI-powered data processing",
    "IoT-based sensor integration",
    "Neural network analysis",
    "Real-time alerts",
    "Smart monitoring"
  ]
}

=== MATCHES ===
[0.49] Intelligent load control system
[0.49] Expandable modularized intelligent socket based on Internet of things
[0.48] An intelligent control platform for IoT data center
[0.48] Novel visual wisdom thing networking gateway equipment
[0.48] An intelligent monitoring device for substation drainage pump based on Internet of Things technology

=== RISK ANALYSIS ===
{
  "risks": [
    {
      "title": "Intelligent load control system",
      "level": "Low",
      "reason": "This patent focuses on intelligent load control within a power grid, using a DSP chip, which is distinct from an AI-powered IoT monitoring system specifically using neural networks for sensor d